In [1]:
import pandas as pd

In [2]:
from pandasql import sqldf

# Define a reusable function for running SQL queries
run_query = lambda query: sqldf(query, globals())

In [3]:
ncbi_histone_df = pd.read_csv("dataset/ncbi_histone_count.csv")

In [4]:
display(ncbi_histone_df.head())
display(ncbi_histone_df.shape)

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k4me3,h3k9ac,h3k27ac,h3k27me3,h3k9me3,histone_count_total
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'",...,none,none,"b'-1,-1,-1,'",11873,1,0,1,1,3,6
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...",...,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,2,2,1,3,1,9
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'",...,none,none,"b'-1,'",17436,0,0,0,0,0,0
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'",...,none,none,"b'-1,'",30365,2,2,1,3,1,9
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'",...,none,none,"b'-1,-1,-1,'",36081,1,1,0,1,0,3


(91315, 23)

In [5]:
hepg2_df = pd.read_csv("dataset/hepg2_exp_transformed.csv")
display(hepg2_df.head())
display(hepg2_df.shape)

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromStart,chromEnd
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090,70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891,328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658,368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585,794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263,843900


(30032, 17)

In [ ]:
# Join the dataset

q_join_hepg2_ncbi = '''
    SELECT h.*, n.*
    FROM hepg2_df h
    JOIN ncbi_histone_df n
    ON h.chrom = n.chrom
    AND h.chromStart = n.txStart
    AND h.chromEnd = n.txEnd
'''

result_1 = run_query(q_join_hepg2_ncbi)

In [ ]:
display(result_1.head())
display(result_1.shape)

In [24]:
q2 = '''
    SELECT  n.name as refseq_name,
            n.chrom refseq_chrom,
            n.txStart as refseq_start_a, n.txEnd refseq_start_b,
            (n.txEnd - n.txStart) as 'refseq_length',
            h.gene_id as hepg2_gene_id,
            h.chrom as hepg2_chrom,
            h.chromStart as hepg2_start_c, h.chromEnd as hep_g2_start_d,
            (h.chromEnd - h.chromStart) as 'hepg2_length',
            (h.chromEnd - n.txStart)/(h.chromEnd - h.chromStart) as 'overlap_ratio'
    FROM hepg2_df h
    JOIN ncbi_histone_df n
    ON h.chrom = n.chrom
    AND (h.chromEnd - n.txStart)/(h.chromEnd - h.chromStart) >= 0.8
    AND h.chromEnd <= n.txEnd
'''

r2 = run_query(q2)

In [25]:
display(r2.head())
display(r2.shape)

,refseq_name,refseq_chrom,refseq_start_a,refseq_start_b,refseq_length,hepg2_gene_id,hepg2_chrom,hepg2_start_c,hep_g2_start_d,hepg2_length,overlap_ratio
0,NM_001005484.2,chr1,65418,71585,6167,XLOC_000001,chr1,69090,70008,918,5
1,NR_028322.1,chr1,323891,328581,4690,XLOC_000002,chr1,323891,328581,4690,1
2,NM_001005221.2,chr1,367658,368597,939,XLOC_000003,chr1,367658,368597,939,1
3,NM_001130045.2,chr1,1109259,1133316,24057,XLOC_000014,chr1,1109285,1133313,24028,1
4,NM_001371649.1,chr1,1109259,1133316,24057,XLOC_000014,chr1,1109285,1133313,24028,1


(12280, 11)

In [18]:
r2[r2["overlap_length"] > 1]

,refseq_start_a,refseq_start_b,hepg2_start_c,hep_g2_start_d,overlap_length
0,65418,71585,69090,70008,5
76,33772366,33786699,33775917,33777393,3
82,38512805,38585202,38575309,38585201,7
89,42921777,43122861,42938423,42939069,26
90,42922213,42939088,42938423,42939069,26
...,...,...,...,...,...
12115,139791923,139869519,139867347,139869363,38
12116,139791923,139873522,139867347,139869363,38
12117,139846761,139873522,139867347,139869363,11
12158,59069570,59115127,59100456,59115123,3
